# Mixture of Experts (MoE)

A standard dense neural network activates **every** parameter for every input. Mixture of Experts
introduces a fundamentally different idea: maintain a collection of specialist sub-networks
("experts") and a lightweight **router** that decides which experts to consult for each input token.

Key concepts:
- **Sparse computation** -- only a subset of the total parameters are active per forward pass.
- **Conditional routing** -- the router learns to send different inputs to different experts.
- **Scaling without proportional cost** -- you can increase total parameters (more experts)
  while keeping per-token compute roughly constant (top-k routing).

This notebook explores our minimal `MoE` layer implementation and builds intuition for
routing behavior, expert specialization, load balancing, and design trade-offs.

## 1. Imports and Setup

In [ ]:
%matplotlib inline

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from moe_layer import MoE

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Core dimensions used throughout the notebook
d = 64          # model / hidden dimension
seq_len = 16    # sequence length (tokens per sample)
batch_size = 2  # number of samples in a batch

print(f"d={d}, seq_len={seq_len}, batch_size={batch_size}")

## 2. Basic MoE Forward Pass

We create an MoE layer with 4 experts and feed it a random input of shape
`(batch_size, seq_len, d)`. The layer returns:
- `output` -- the weighted combination of all expert outputs, same shape as input.
- `router_weights` -- softmax probabilities over experts for every token.

In [ ]:
torch.manual_seed(42)
moe = MoE(d, experts=4)
x = torch.randn(batch_size, seq_len, d)

output, router_weights = moe(x)

print("Input shape: ", x.shape)
print("Output shape:", output.shape)
print("Router weights shape:", router_weights.shape)

In [ ]:
# The output has the same shape as the input -- MoE is a drop-in replacement
# for any feed-forward layer.
assert output.shape == x.shape
print("Output tensor (first token of first sample):")
print(output[0, 0, :8].detach())  # first 8 dims

## 3. Inspecting Router Weights

The router assigns a probability to each expert for every token. Let's look at the
raw numbers to see which experts the router prefers.

In [ ]:
# Router weights for sample 0, all tokens
rw = router_weights[0].detach()  # (seq_len, num_experts)
print("Router weights for sample 0 (tokens x experts):")
print(rw.numpy().round(3))

In [ ]:
# Which expert gets the highest weight for each token?
top_expert = rw.argmax(dim=-1)
print("Top expert per token:", top_expert.numpy())

## 4. Visualizing Router Weights as a Heatmap

A heatmap lets us see at a glance how the router distributes load across experts
for every token in the sequence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx in range(batch_size):
    rw_np = router_weights[idx].detach().numpy()
    im = axes[idx].imshow(rw_np, aspect='auto', cmap='YlOrRd')
    axes[idx].set_xlabel('Expert')
    axes[idx].set_ylabel('Token position')
    axes[idx].set_title(f'Sample {idx}')
    axes[idx].set_xticks(range(rw_np.shape[1]))
    plt.colorbar(im, ax=axes[idx], shrink=0.8)

plt.suptitle('Router Weight Heatmaps (tokens x experts)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Expert Specialization

Does the router send different kinds of inputs to different experts? We generate
multiple batches and accumulate statistics on which expert is selected most often.

In [ ]:
torch.manual_seed(42)
num_batches = 50
num_experts = 4
expert_counts = torch.zeros(num_experts)
expert_total_weight = torch.zeros(num_experts)

for _ in range(num_batches):
    x_i = torch.randn(batch_size, seq_len, d)
    _, rw_i = moe(x_i)
    rw_i = rw_i.detach()
    # Count top-1 selections
    top1 = rw_i.argmax(dim=-1).flatten()
    for e in range(num_experts):
        expert_counts[e] += (top1 == e).sum()
    # Accumulate total weight mass
    expert_total_weight += rw_i.sum(dim=(0, 1))

print("Top-1 selection counts per expert:", expert_counts.numpy().astype(int))
print("Total weight mass per expert:     ", expert_total_weight.numpy().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(num_experts), expert_counts.numpy(), color='steelblue')
axes[0].set_xlabel('Expert')
axes[0].set_ylabel('Top-1 selection count')
axes[0].set_title('Expert Selection Frequency')
axes[0].set_xticks(range(num_experts))

axes[1].bar(range(num_experts), expert_total_weight.numpy(), color='coral')
axes[1].set_xlabel('Expert')
axes[1].set_ylabel('Cumulative weight')
axes[1].set_title('Total Router Weight Mass')
axes[1].set_xticks(range(num_experts))

plt.tight_layout()
plt.show()

## 6. Expert Utilization -- Is the Load Balanced?

Ideally each expert should handle roughly the same fraction of tokens. If one expert
dominates, the others waste parameters. We measure balance with a coefficient of
variation (CV) and routing entropy.

In [ ]:
fractions = expert_counts / expert_counts.sum()
ideal = 1.0 / num_experts

cv = fractions.std() / fractions.mean()  # coefficient of variation
print(f"Expert load fractions: {fractions.numpy().round(4)}")
print(f"Ideal uniform fraction: {ideal:.4f}")
print(f"Coefficient of variation: {cv.item():.4f}  (0 = perfect balance)")

In [ ]:
# Routing entropy: higher = more uniform distribution
# Maximum entropy for 4 experts is log(4) ~ 1.386
mean_weights = expert_total_weight / expert_total_weight.sum()
entropy = -(mean_weights * torch.log(mean_weights + 1e-8)).sum()
max_entropy = np.log(num_experts)

print(f"Routing entropy: {entropy.item():.4f}")
print(f"Maximum entropy: {max_entropy:.4f}")
print(f"Normalized entropy (1.0 = perfect): {entropy.item() / max_entropy:.4f}")

## 7. Parameter Count: Dense vs. MoE

A key MoE selling point is that you can increase total parameters without proportionally
increasing compute. Let's compare a single dense layer to an MoE layer with equivalent
total parameter budget.

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

dense_layer = nn.Linear(d, d)
moe_4 = MoE(d, experts=4)
moe_8 = MoE(d, experts=8)

print(f"Dense Linear({d},{d}):  {count_params(dense_layer):>8,} params")
print(f"MoE(d={d}, experts=4):  {count_params(moe_4):>8,} params")
print(f"MoE(d={d}, experts=8):  {count_params(moe_8):>8,} params")
print(f"\nMoE-4 has {count_params(moe_4) / count_params(dense_layer):.1f}x the params of a dense layer")
print(f"MoE-8 has {count_params(moe_8) / count_params(dense_layer):.1f}x the params of a dense layer")

In [ ]:
# Break down where the parameters live
router_params = count_params(moe_4.router)
expert_params = sum(count_params(e) for e in moe_4.experts)
print(f"MoE-4 breakdown:")
print(f"  Router params:  {router_params:>6,}  ({100*router_params/count_params(moe_4):.1f}%)")
print(f"  Expert params:  {expert_params:>6,}  ({100*expert_params/count_params(moe_4):.1f}%)")
print(f"  Total:          {count_params(moe_4):>6,}")

## 8. Active Parameters per Token

In soft routing (our implementation), every expert is technically active. But most of
the weight mass concentrates on just a few experts. We quantify the "effective" number
of active experts per token.

In [ ]:
torch.manual_seed(42)
x_test = torch.randn(1, seq_len, d)
_, rw_test = moe_4(x_test)
rw_test = rw_test.detach().squeeze(0)  # (seq_len, 4)

# Effective number of experts = exp(entropy) -- the perplexity of the distribution
per_token_entropy = -(rw_test * torch.log(rw_test + 1e-8)).sum(dim=-1)
effective_experts = torch.exp(per_token_entropy)

print("Effective # of active experts per token:")
for t in range(seq_len):
    weights_str = ", ".join(f"{w:.3f}" for w in rw_test[t].numpy())
    print(f"  Token {t:2d}: weights=[{weights_str}]  effective={effective_experts[t]:.2f}")

In [ ]:
print(f"\nMean effective experts: {effective_experts.mean().item():.2f} out of 4")
print(f"If all weight on 1 expert: effective = 1.00")
print(f"If perfectly uniform:      effective = 4.00")

## 9. Ablation: Number of Experts

How does the number of experts affect output diversity and routing behavior?
We test with 2, 4, and 8 experts.

In [ ]:
expert_counts_list = [2, 4, 8]
results = {}

x_fixed = torch.randn(batch_size, seq_len, d)

for n_exp in expert_counts_list:
    torch.manual_seed(42)
    moe_test = MoE(d, experts=n_exp)
    out_test, rw_test = moe_test(x_fixed)
    rw_det = rw_test.detach()

    # Output diversity: std of output across tokens
    output_std = out_test.detach().std().item()

    # Routing entropy per token, averaged
    ent = -(rw_det * torch.log(rw_det + 1e-8)).sum(dim=-1).mean().item()
    max_ent = np.log(n_exp)

    results[n_exp] = {
        'output_std': output_std,
        'mean_entropy': ent,
        'max_entropy': max_ent,
        'normalized_entropy': ent / max_ent,
        'router_weights': rw_det,
    }

    print(f"Experts={n_exp}: output_std={output_std:.4f}, "
          f"routing_entropy={ent:.4f}/{max_ent:.4f} "
          f"(normalized={ent/max_ent:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, n_exp in enumerate(expert_counts_list):
    rw_np = results[n_exp]['router_weights'][0].numpy()  # sample 0
    im = axes[i].imshow(rw_np, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    axes[i].set_xlabel('Expert')
    axes[i].set_ylabel('Token position')
    axes[i].set_title(f'{n_exp} experts (norm. ent. = {results[n_exp]["normalized_entropy"]:.3f})')
    axes[i].set_xticks(range(n_exp))
    plt.colorbar(im, ax=axes[i], shrink=0.8)

plt.suptitle('Router Weights for Different Expert Counts', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Summary bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar([str(n) for n in expert_counts_list],
            [results[n]['output_std'] for n in expert_counts_list],
            color='steelblue')
axes[0].set_xlabel('Number of experts')
axes[0].set_ylabel('Output std')
axes[0].set_title('Output Diversity')

axes[1].bar([str(n) for n in expert_counts_list],
            [results[n]['normalized_entropy'] for n in expert_counts_list],
            color='coral')
axes[1].set_xlabel('Number of experts')
axes[1].set_ylabel('Normalized entropy')
axes[1].set_title('Routing Entropy (1.0 = uniform)')
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## 10. Top-k Routing

Real MoE architectures (Switch Transformer, Mixtral) don't use soft routing over
all experts. Instead they select the **top-k** experts (often k=1 or k=2) and zero
out the rest. This makes computation truly sparse.

Let's implement top-k selection on top of our soft router output and compare.

In [ ]:
def topk_route(router_weights, k=2):
    """Zero out all but the top-k experts, then renormalize."""
    topk_vals, topk_idx = router_weights.topk(k, dim=-1)
    mask = torch.zeros_like(router_weights)
    mask.scatter_(-1, topk_idx, 1.0)
    masked_weights = router_weights * mask
    # Renormalize so weights sum to 1
    masked_weights = masked_weights / (masked_weights.sum(dim=-1, keepdim=True) + 1e-8)
    return masked_weights

# Test with our 4-expert MoE
torch.manual_seed(42)
moe_4 = MoE(d, experts=4)
x_test = torch.randn(1, seq_len, d)
_, rw_soft = moe_4(x_test)
rw_soft = rw_soft.detach()

rw_top2 = topk_route(rw_soft, k=2)
rw_top1 = topk_route(rw_soft, k=1)

print("Soft routing (token 0):", rw_soft[0, 0].numpy().round(3))
print("Top-2 routing (token 0):", rw_top2[0, 0].numpy().round(3))
print("Top-1 routing (token 0):", rw_top1[0, 0].numpy().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (label, rw) in zip(axes, [
    ('Soft (all experts)', rw_soft[0]),
    ('Top-2', rw_top2[0]),
    ('Top-1', rw_top1[0]),
]):
    im = ax.imshow(rw.numpy(), aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xlabel('Expert')
    ax.set_ylabel('Token position')
    ax.set_title(label)
    ax.set_xticks(range(4))
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Soft vs. Top-k Routing', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Compute top-k output manually to compare with soft output
def moe_forward_topk(moe_layer, x, k):
    """Run MoE with top-k routing applied post-hoc."""
    w = torch.softmax(moe_layer.router(x), dim=-1)
    w_topk = topk_route(w, k=k)
    experts = moe_layer.experts
    out = sum(w_topk[..., i:i+1] * experts[i](x) for i in range(len(experts)))
    return out, w_topk

out_soft, _ = moe_4(x_test)
out_top2, _ = moe_forward_topk(moe_4, x_test, k=2)
out_top1, _ = moe_forward_topk(moe_4, x_test, k=1)

diff_top2 = (out_soft - out_top2).abs().mean().item()
diff_top1 = (out_soft - out_top1).abs().mean().item()
print(f"Mean absolute difference from soft routing:")
print(f"  Top-2: {diff_top2:.6f}")
print(f"  Top-1: {diff_top1:.6f}")

## 11. The Load Balancing Problem

A well-known challenge with MoE is **expert collapse**: the router learns to send most
tokens to one or two experts, leaving the rest underutilized. This wastes capacity.

We simulate this by biasing the router and then show how an **auxiliary load-balancing
loss** can encourage uniform utilization.

In [ ]:
# Create a biased MoE by manually pushing the router bias
torch.manual_seed(42)
moe_biased = MoE(d, experts=4)

# Artificially make expert 0 much more preferred
with torch.no_grad():
    moe_biased.router.bias[0] += 5.0

_, rw_biased = moe_biased(x_fixed)
rw_biased = rw_biased.detach()

print("Biased router weights (first 4 tokens, sample 0):")
print(rw_biased[0, :4].numpy().round(4))
print(f"\nExpert 0 gets {rw_biased[..., 0].mean().item()*100:.1f}% of total weight")

In [ ]:
# Visualize the imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mean_load_biased = rw_biased.mean(dim=(0, 1)).numpy()
mean_load_normal = router_weights.detach().mean(dim=(0, 1)).numpy()

axes[0].bar(range(4), mean_load_normal, color='steelblue')
axes[0].axhline(y=0.25, color='red', linestyle='--', label='Ideal')
axes[0].set_title('Normal Router')
axes[0].set_xlabel('Expert')
axes[0].set_ylabel('Mean weight')
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].bar(range(4), mean_load_biased, color='coral')
axes[1].axhline(y=0.25, color='red', linestyle='--', label='Ideal')
axes[1].set_title('Biased Router (expert collapse)')
axes[1].set_xlabel('Expert')
axes[1].set_ylabel('Mean weight')
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.show()

## 12. Auxiliary Load-Balancing Loss

The standard remedy (used in Switch Transformer and GShard) is to add an auxiliary loss
that penalizes uneven expert utilization:

$$\mathcal{L}_{\text{balance}} = N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ (top-1) and $P_i$ is the
mean router probability for expert $i$. This loss is minimized when all experts get
equal load.

In [ ]:
def load_balance_loss(router_weights):
    """
    Compute the auxiliary load-balancing loss.
    router_weights: (batch, seq_len, num_experts) -- softmax probabilities.
    """
    num_experts = router_weights.shape[-1]
    # f_i: fraction of tokens where expert i is the top-1 choice
    top1 = router_weights.argmax(dim=-1)  # (batch, seq_len)
    f = torch.zeros(num_experts, device=router_weights.device)
    total_tokens = top1.numel()
    for i in range(num_experts):
        f[i] = (top1 == i).float().sum() / total_tokens

    # P_i: mean router probability for expert i across all tokens
    P = router_weights.mean(dim=(0, 1))  # (num_experts,)

    # Loss: N * sum(f_i * P_i)
    loss = num_experts * (f * P).sum()
    return loss

loss_normal = load_balance_loss(router_weights.detach())
loss_biased = load_balance_loss(rw_biased)

print(f"Load balance loss (normal router):  {loss_normal.item():.4f}")
print(f"Load balance loss (biased router):  {loss_biased.item():.4f}")
print(f"Ideal loss (perfectly balanced):     {1.0:.4f}")
print(f"\nHigher loss = worse balance. The biased router gets a much higher penalty.")

In [ ]:
# Demonstrate training with load-balancing loss to fix the biased router
torch.manual_seed(42)
moe_fix = MoE(d, experts=4)
with torch.no_grad():
    moe_fix.router.bias[0] += 5.0  # start biased

optimizer = torch.optim.Adam(moe_fix.parameters(), lr=0.01)
balance_coeff = 0.1  # weight for the auxiliary loss

loss_history = []
entropy_history = []

for step in range(200):
    x_train = torch.randn(batch_size, seq_len, d)
    out, rw = moe_fix(x_train)

    # Primary loss: just MSE toward zero (dummy task)
    primary_loss = out.pow(2).mean()
    # Auxiliary balance loss
    aux_loss = load_balance_loss(rw)
    total_loss = primary_loss + balance_coeff * aux_loss

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    # Track balance
    with torch.no_grad():
        rw_det = rw.detach()
        ent = -(rw_det * torch.log(rw_det + 1e-8)).sum(dim=-1).mean().item()
        loss_history.append(aux_loss.item())
        entropy_history.append(ent)

print(f"Aux loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}")
print(f"Entropy:  {entropy_history[0]:.3f} -> {entropy_history[-1]:.3f} (max={np.log(4):.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history, color='coral')
axes[0].axhline(y=1.0, color='green', linestyle='--', label='Ideal')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Balance loss')
axes[0].set_title('Load-Balancing Loss During Training')
axes[0].legend()

axes[1].plot(entropy_history, color='steelblue')
axes[1].axhline(y=np.log(4), color='green', linestyle='--', label='Max entropy')
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Routing entropy')
axes[1].set_title('Routing Entropy During Training')
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Router Behavior on Diverse Inputs

Let's see how the trained (balanced) router distributes a batch of diverse inputs
across experts.

In [ ]:
torch.manual_seed(0)
x_diverse = torch.randn(8, seq_len, d)  # 8 different samples
_, rw_diverse = moe_fix(x_diverse)
rw_diverse = rw_diverse.detach()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i in range(8):
    im = axes[i].imshow(rw_diverse[i].numpy(), aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    axes[i].set_title(f'Sample {i}')
    axes[i].set_xlabel('Expert')
    axes[i].set_xticks(range(4))
    if i % 4 == 0:
        axes[i].set_ylabel('Token position')

plt.suptitle('Router Weights Across 8 Diverse Inputs (trained with balance loss)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Aggregate: mean expert weight across all diverse samples
mean_per_expert = rw_diverse.mean(dim=(0, 1)).numpy()

plt.figure(figsize=(6, 4))
plt.bar(range(4), mean_per_expert, color='steelblue')
plt.axhline(y=0.25, color='red', linestyle='--', label='Ideal uniform')
plt.xlabel('Expert')
plt.ylabel('Mean router weight')
plt.title('Expert Load Across 8 Diverse Samples')
plt.xticks(range(4))
plt.legend()
plt.tight_layout()
plt.show()

## 14. MoE vs. Dense Output Distribution

How does the output distribution of an MoE layer compare to a single dense layer
of the same hidden dimension? The MoE output is a *mixture* of several linear
transforms, so its distribution can be more complex.

In [ ]:
torch.manual_seed(42)
dense = nn.Linear(d, d)
moe_compare = MoE(d, experts=4)

x_compare = torch.randn(32, seq_len, d)  # larger batch

with torch.no_grad():
    out_dense = dense(x_compare)
    out_moe, _ = moe_compare(x_compare)

print(f"Dense output -- mean: {out_dense.mean():.4f}, std: {out_dense.std():.4f}")
print(f"MoE output   -- mean: {out_moe.mean():.4f}, std: {out_moe.std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(out_dense.flatten().numpy(), bins=80, alpha=0.7, color='steelblue',
             density=True, label='Dense')
axes[0].hist(out_moe.flatten().numpy(), bins=80, alpha=0.7, color='coral',
             density=True, label='MoE')
axes[0].set_xlabel('Activation value')
axes[0].set_ylabel('Density')
axes[0].set_title('Output Value Distribution')
axes[0].legend()

# Per-token output norm comparison
norms_dense = out_dense.flatten(0, 1).norm(dim=-1).numpy()
norms_moe = out_moe.flatten(0, 1).norm(dim=-1).numpy()

axes[1].hist(norms_dense, bins=40, alpha=0.7, color='steelblue',
             density=True, label='Dense')
axes[1].hist(norms_moe, bins=40, alpha=0.7, color='coral',
             density=True, label='MoE')
axes[1].set_xlabel('L2 norm')
axes[1].set_ylabel('Density')
axes[1].set_title('Per-Token Output Norm Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 15. Key Insights

### When does MoE help?
- **Scaling model capacity without proportional compute increase.** MoE lets you add
  parameters (more experts) while keeping per-token FLOPs constant if top-k routing is used.
- **Diverse inputs benefit from specialization.** When different inputs require
  different processing, dedicated experts can learn specialized transformations.
- **Large-scale training.** MoE shows the greatest benefit when the model is large
  enough that individual experts can specialize meaningfully.

### The load balancing challenge
- Without intervention, routers tend toward **expert collapse** -- routing most tokens
  to a small subset of experts.
- An **auxiliary load-balancing loss** is the standard mitigation. It encourages the
  router to distribute tokens more uniformly across experts.
- The coefficient for this loss is a sensitive hyperparameter: too low and collapse
  persists; too high and routing becomes uniform regardless of input (losing
  specialization benefits).

### Real-world MoE architectures
- **Switch Transformer** (Google, 2021): Uses top-1 routing with a capacity factor to
  limit how many tokens each expert processes. Achieved the same quality as T5 with
  significantly less training compute.
- **GShard** (Google, 2020): Top-2 routing with expert parallelism across TPU pods.
- **Mixtral 8x7B** (Mistral, 2024): 8 experts with top-2 routing. Each token uses
  2 of 8 experts (12.9B active params out of 46.7B total). Matches or exceeds
  Llama 2 70B while being much cheaper to run.
- **DeepSeek-V2** (2024): Fine-grained expert segmentation with shared experts
  and routed experts, enabling efficient inference at scale.

In [ ]:
# Final summary: our MoE layer at a glance
print("=" * 60)
print("MoE Layer Summary")
print("=" * 60)
print(f"Hidden dimension:       {d}")
print(f"Number of experts:      4")
print(f"Router type:            Learned linear -> softmax")
print(f"Routing:                Soft (all experts weighted)")
print(f"Total params (MoE-4):   {count_params(MoE(d, experts=4)):,}")
print(f"Total params (dense):   {count_params(nn.Linear(d, d)):,}")
print(f"Ratio:                  {count_params(MoE(d, experts=4)) / count_params(nn.Linear(d, d)):.1f}x")
print("=" * 60)